<a href="https://colab.research.google.com/github/psehgal2/Pandemaniac/blob/main/Strategy5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import networkx as nx
import numpy as np
import networkx as nx
import numpy as np
import math
from networkx.algorithms.community import greedy_modularity_communities
import random


class CompetingCascade:
    def __init__(self, G, format, seeds, id):
        self.graph = G
        self.num_nodes = G.number_of_nodes()
        self.num_seeds = seeds # Set the value of N when the object is created
        self.iter = 1
        self.means = np.zeros(self.num_nodes)
        self.times = np.ones(self.num_nodes)  # Start with one to avoid division by zero
        adj_arr = nx.to_numpy_array(G)
        adj_list = adjacency_matrix_to_dict(adj_arr)
        self.adj_list = adj_list
        self.ts = ThomsponSampling(G, format, seeds, id, adj_list)
        self.betweenness_centrality = dict(nx.eigenvector_centrality(G))
        self.clusters = list(nx.connected_components(G))

    def max_degree(self):
        degrees = dict(self.graph.degree())
        nodes = sorted(degrees, key=degrees.get, reverse=True)
        return nodes[:self.num_seeds]

    def competing_strategy(self):
      # check if the graph is connnecteddd
        if not nx.is_connected(self.graph):
            print("Graph is not connected.")
            return

        # get the communities
        n_communities = list(greedy_modularity_communities(self.graph))
        avg_degrees = {community: sum(self.graph.degree(node) for node in community) / len(community) for community in n_communities}
        sorted_communities = sorted(avg_degrees.items(), reverse=True, key=lambda val: val[1])

        k = 9
        largest_comm = sorted_communities[:k]

        if not largest_comm:
          print("No communities were identified")
          return

        nodes_per_clust = math.ceil(self.num_seeds/k)
        print("Nodes per cluster: ", nodes_per_clust)

        selected_nodes = []

        for i, (cluster, _) in enumerate(largest_comm):
            clustering_subgraph = self.graph.subgraph(cluster)
            degrees = dict(clustering_subgraph.degree())
            sorted_nodes = sorted(degrees, key=degrees.get, reverse=True)
            n_sorted = sorted_nodes[:nodes_per_clust]
            selected_nodes.extend(n_sorted)

        random.shuffle(selected_nodes)
        return selected_nodes[:self.num_seeds]

    def update_parameters(self, selected_nodes, node_conquer_count):
        # Update parameters based on obtained rewards
        for node in selected_nodes:
            self.times[node] += 1
            self.means[node] += (node_conquer_count[str(node)] - self.means[node]) / self.times[node]

    def main_loop(self):
        # Thompson Sampling iterations
        num_iterations = 1000
        for _ in range(num_iterations):
            # Combined strategy to select the top k nodes
            selected_nodes = self.competing_strategy()
            selected_nodes_str = map(str, selected_nodes)

            opp_nodes = self.ts.thompsons_sampling_new()
            opp_nodes_str = map(str, opp_nodes)

            node_mappings = {'self': selected_nodes_str, 'opp_strat': opp_nodes_str}
            overall_result, node_conquer_count = run_simulations_2(self.adj_list, node_mappings)

            # Update Thompson Sampling parameters based on the obtained rewards
            self.update_parameters(selected_nodes, node_conquer_count)

        # Final seed nodes after Thompson Sampling iterations
        final_seed_nodes = self.thompsons_sampling()

        return final_seed_nodes

In [ ]:
class GraphProcessing:
    def __init__(self, filepath):
        self.filepath = filepath
        self.graph = None
        self.format = None
        self.num_seeds = None
        self.unique_id = None
        self.adjacency = None

    def get_graph(self):
        return self.graph

    def get_format(self):
        return self.format

    def get_num_seeds(self):
        return self.num_seeds

    def get_unique_id(self):
        return self.unique_id

    def get_adjacency(self):
      return self.adjacency

    def open_sampling_file(self):
        components = self.filepath.split('.')
        competition_format = components[0]
        num_seeds = int(components[1])
        unique_id = int(components[2])

        with open(file_path, "r") as file:
            file_contents = json.load(file)
        return file_contents, competition_format, num_seeds, unique_id

    def convert_to_graph(self):
        adjacency, competition_format, num_seeds, unique_id = self.open_sampling_file()
        self.adjacency = adjacency
        G = nx.Graph(adjacency)
        assert len(adjacency) == nx.number_of_nodes(G)
        self.graph = G
        self.format = competition_format
        self.num_seeds = num_seeds
        self.unique_id = unique_id

In [ ]:
'''
===========
   USAGE
===========

>>> import sim
>>> sim.run([graph], [dict with keys as names and values as a list of nodes])

Returns a dictionary containing the names and the number of nodes they got.

Example:
>>> graph = {"2": ["6", "3", "7", "2"], "3": ["2", "7, "12"], ... }
>>> nodes = {"strategy1": ["1", "5"], "strategy2": ["5", "23"], ... }
>>> sim.run(graph, nodes)
>>> {"strategy1": 243, "strategy6": 121, "strategy2": 13}

Possible Errors:
- KeyError: Will occur if any seed nodes are invalid (i.e. do not exist on the
            graph).
'''

from collections import Counter, OrderedDict
from copy import deepcopy
from random import randint


def run(adj_list, node_mappings):
  """
  Function: run
  -------------
  Runs the simulation on a graph with the given node mappings.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  results = run_simulation(adj_list, node_mappings)
  return results

def run_simulations_2(adj_list, node_mappings):
    """
    Function: run_simulation
    ------------------------
    Runs the simulation. Returns a tuple with the overall result and a dictionary
    with the key as the "color"/name, and the value as the number of nodes that
    "color"/name got.

    adj_list: A dictionary representation of the graph adjacencies.
    node_mappings: A dictionary where the key is a name and the value is a list
                   of seed nodes associated with that name.
    """
    # Stores a mapping of nodes to their color.
    node_color = dict([(node, None) for node in adj_list.keys()])
    init(node_mappings, node_color)
    generation = 1

    # Keep calculating the epidemic until it stops changing. Randomly choose
    # number between 100 and 200 as the stopping point if the epidemic does not
    # converge.
    prev = None
    nodes = adj_list.keys()
    max_rounds = randint(100, 200)

    # Track the number of nodes conquered by each node
    node_conquer_count = dict([(node, 0) for node in adj_list.keys()])

    while not is_stable(generation, max_rounds, prev, node_color):
        prev = deepcopy(node_color)
        for node in nodes:
            (changed, color) = update(adj_list, prev, node)
            # Store the node's new color only if it changed.
            if changed:
                node_color[node] = color
                # Increment the conquer count for the node
                node_conquer_count[node] += 1

        # NOTE: prev contains the state of the graph of the previous generation,
        # node_colors contains the state of the graph at the current generation.
        # You could check these two dicts if you want to see the intermediate steps
        # of the epidemic.
        generation += 1

    # Return both the overall result and individual node contributions
    return get_result(node_mappings.keys(), node_color), node_conquer_count


def run_simulation(adj_list, node_mappings):
  """
  Function: run_simulation
  ------------------------
  Runs the simulation. Returns a dictionary with the key as the "color"/name,
  and the value as the number of nodes that "color"/name got.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  # Stores a mapping of nodes to their color.
  node_color = dict([(node, None) for node in adj_list.keys()])
  init(node_mappings, node_color)
  generation = 1

  # Keep calculating the epidemic until it stops changing. Randomly choose
  # number between 100 and 200 as the stopping point if the epidemic does not
  # converge.
  prev = None
  nodes = adj_list.keys()
  max_rounds = randint(100, 200)
  while not is_stable(generation, max_rounds, prev, node_color):
    prev = deepcopy(node_color)
    for node in nodes:
      (changed, color) = update(adj_list, prev, node)
      # Store the node's new color only if it changSed.
      if changed: node_color[node] = color
    # NOTE: prev contains the state of the graph of the previous generation,
    # node_colros contains the state of the graph at the current generation.
    # You could check these two dicts if you want to see the intermediate steps
    # of the epidemic.
    generation += 1

  return get_result(node_mappings.keys(), node_color)


def init(color_nodes, node_color):
  """
  Function: init
  --------------
  Initializes the node to color mappings.
  """
  for (color, nodes) in color_nodes.items():
    for node in nodes:
      if node_color[node] is not None:
        node_color[node] = "__CONFLICT__"
      else:
        node_color[node] = color
  for (node, color) in node_color.items():
    if color == "__CONFLICT__":
      node_color[node] = None


def update(adj_list, node_color, node):
  """
  Function: update
  ----------------
  Updates each node based on its neighbors.
  """
  neighbors = adj_list[node]
  colored_neighbors = list(filter(None, [node_color[x] for x in neighbors]))
  total_votes = len(colored_neighbors)
  team_count = Counter(colored_neighbors)
  if node_color[node] is not None:
    team_count[node_color[node]] += 1.5
    total_votes += 1.5
  most_common = team_count.most_common(1)
  if len(most_common) > 0 and \
    most_common[0][1] > total_votes / 2.0:
    return (True, most_common[0][0])

  return (False, node_color[node])


def is_stable(generation, max_rounds, prev, curr):
  """
  Function: is_stable
  -------------------
  Checks whether or not the epidemic has stabilized.
  """
  if generation <= 1 or prev is None:
    return False
  if generation == max_rounds:
    return True
  for node, color in curr.items():
    if not prev[node] == curr[node]:
      return False
  return True


def get_result(colors, node_color):
  """
  Function: get_result
  --------------------
  Get the resulting mapping of colors to the number of nodes of that color.
  """
  color_nodes = {}
  for color in colors:
    color_nodes[color] = 0
  for node, color in node_color.items():
    if color is not None:
      color_nodes[color] += 1
  return color_nodes



In [ ]:
file_path = "RR.10.51.json"
g = GraphProcessing(file_path)
g.convert_to_graph()

G = g.get_graph()
print(G.number_of_nodes())
print("Edges:", len(G.edges))
print ("clustering of G is:", nx.transitivity(G))
format = g.get_format()
k = g.get_num_seeds()
id = g.get_unique_id()
adj_list = g.get_adjacency()
# ts1 = ThompsonSampling1(G, format, k, id)
# ts1_nodes = ts1.main_loop()
# ts1_nodes_str = map(str, ts1_nodes)
# ts = ThomsponSampling(G, format, k, id, adj_list)
# ts_nodes = ts.thompsons_sampling_new()
# ts_nodes_str = map(str, ts_nodes)
ts_nodes_str = ["23", "148", "163", "55", "145", "109", "22", "107", "139", "24"]
cc = CompetingCascade(G, format, k, id)
cc_nodes = cc.competing_strategy()
cc_nodes_str = map(str, cc_nodes)
nodes = {"self": cc_nodes_str, "old_ts": ts_nodes_str}
adjacency = g.get_adjacency()
print("Run: ", run(adjacency, nodes))

FileNotFoundError: [Errno 2] No such file or directory: 'RR.10.51.json'